In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torch import nn, optim

In [2]:
dataset_path = r'dataset\raw-img'

In [3]:


dataset = datasets.ImageFolder(root=dataset_path,
                               transform=transforms.Compose([
                                   transforms.Resize((96, 96)),
                                   transforms.ToTensor(),
                                   transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]))

loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)

In [4]:
images, labels = next(iter(loader))
print(images.shape)print(labels)

torch.Size([32, 3, 96, 96])
tensor([8, 6, 0, 9, 6, 7, 0, 4, 5, 6, 0, 9, 9, 8, 3, 0, 9, 8, 1, 6, 3, 0, 5, 1,
        8, 0, 3, 4, 2, 3, 7, 3])


In [5]:
print(dataset.class_to_idx)

{'cane': 0, 'cavallo': 1, 'elefante': 2, 'farfalla': 3, 'gallina': 4, 'gatto': 5, 'mucca': 6, 'pecora': 7, 'ragno': 8, 'scoiattolo': 9}


In [6]:
class Centralized(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                         
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                        
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                         
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                         
            nn.Conv2d(256, 512, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                         
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 3 * 3, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Centralized().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 15
for epoch in range(num_epochs):
   
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        print(loss)
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'Train Loss: {train_loss/len(train_loader):.4f} | Acc: {100.*correct/total:.2f}%')
    print(f'Val Loss: {val_loss/len(val_loader):.4f} | Acc: {100.*val_correct/val_total:.2f}%')
    print('-' * 50)

tensor(2.3116, grad_fn=<NllLossBackward0>)
tensor(2.2990, grad_fn=<NllLossBackward0>)
tensor(2.2521, grad_fn=<NllLossBackward0>)
tensor(2.1891, grad_fn=<NllLossBackward0>)
tensor(2.5608, grad_fn=<NllLossBackward0>)
tensor(2.2468, grad_fn=<NllLossBackward0>)
tensor(2.2728, grad_fn=<NllLossBackward0>)
tensor(2.2819, grad_fn=<NllLossBackward0>)
tensor(2.2812, grad_fn=<NllLossBackward0>)
tensor(2.2837, grad_fn=<NllLossBackward0>)
tensor(2.2569, grad_fn=<NllLossBackward0>)
tensor(2.2643, grad_fn=<NllLossBackward0>)
tensor(2.2589, grad_fn=<NllLossBackward0>)
tensor(2.1855, grad_fn=<NllLossBackward0>)
tensor(2.1773, grad_fn=<NllLossBackward0>)
tensor(2.2165, grad_fn=<NllLossBackward0>)
tensor(2.2465, grad_fn=<NllLossBackward0>)
tensor(2.2190, grad_fn=<NllLossBackward0>)
tensor(2.2362, grad_fn=<NllLossBackward0>)
tensor(2.2455, grad_fn=<NllLossBackward0>)
tensor(2.2559, grad_fn=<NllLossBackward0>)
tensor(2.2455, grad_fn=<NllLossBackward0>)
tensor(2.2181, grad_fn=<NllLossBackward0>)
tensor(2.18